In [ ]:
'''
This notebook calibrates Sentinel-2 L1b data from digital counts into
radiance units, and then merges the files from each MSI detector into
one per-band image.

This script was originally intended for internal use and may have
limited functionality, bugs and other problems. It does not have 
detailed documentation. Please contact the ESA Sentinel-2 Next Generation 
mission scientist for more information: simon{dot}proud{at}esa{dot}int

Please note, some directory structures in this notebook are
hardcoded and will require modification. Check the whole notebook
carefully before running on your system.

Lastly, some of the helper functions in this script were written with
the assistance of Copilot and are definitely not optimised. Pull requests
to streamline the code or add new features are very much welcome!
'''

from __future__ import annotations

from concurrent.futures import ProcessPoolExecutor, as_completed
from rasterio.merge import merge as rio_merge
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from glob import glob
import numpy as np
import rasterio
import tempfile
import re
import os
import traceback

@dataclass(frozen=True)
class BandCal:
    gain: float
    add_offset: float
        
def _safe_open_write(path: Path):
    if path.exists():
        return False
    return True

def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag

# Find the per-band gain and offset values needed for calibration
def read_s2_l1b_ds_band_calibration(mtd_xml):
    root = ET.parse(Path(mtd_xml)).getroot()

    gains = {}
    offsets = {}

    for e in root.iter():
        if _strip_ns(e.tag) == "Spectral_Band_Information" and "bandId" in e.attrib:
            band_id = int(e.attrib["bandId"])
            for c in e:
                if _strip_ns(c.tag) == "PHYSICAL_GAINS":
                    txt = (c.text or "").strip()
                    if not txt:
                        raise RuntimeError(f"Empty PHYSICAL_GAINS for bandId={band_id}")
                    gains[band_id] = float(txt)
                    
    for e in root.iter():
        if _strip_ns(e.tag) == "RADIO_ADD_OFFSET" and "band_id" in e.attrib:
            band_id = int(e.attrib["band_id"])
            txt = (e.text or "").strip()
            if not txt:
                raise RuntimeError(f"Empty RADIO_ADD_OFFSET for band_id={band_id}")
            offsets[band_id] = float(txt)

    if not gains:
        raise RuntimeError("No PHYSICAL_GAINS found in DS metadata XML")
    if not offsets:
        raise RuntimeError("No RADIO_ADD_OFFSET entries found in DS metadata XML")

    out = {}
    for band_id, gain in gains.items():
        code = BAND_ID_TO_CODE.get(band_id)
        if code is None:
            continue
        if band_id not in offsets:
            raise RuntimeError(f"Missing RADIO_ADD_OFFSET for bandId={band_id} ({code})")
        out[code] = BandCal(gain=gain, add_offset=offsets[band_id])

    return out

def parse_band_det(tif_path):
    m = NAME_RX.search(tif_path.name)
    if not m:
        raise ValueError(f"Filename does not match '*_D??_B??.tif': {tif_path.name}")
    return f"B{m.group('band')}".upper(), f"D{m.group('det')}".upper()

def collect_inputs_by_band(input_dir):
    in_dir = Path(input_dir)
    by_band = {}

    for p in sorted(in_dir.glob("*.tif*")):
        if not NAME_RX.search(p.name):
            continue
        band, det = parse_band_det(p)
        by_band.setdefault(band, []).append((det, p))

    if not by_band:
        raise FileNotFoundError(f"No files matching '*_D??_B??.tif' found in {in_dir}")

    return by_band

# Convert from digital count into radiance
def calibrate_geotiff(src_path, dst_path, cal, tiled=False, tile_size=512):
    if tiled and tile_size % 16:
        raise ValueError("tile_size must be a multiple of 16")

    with rasterio.open(src_path) as src:
        profile = src.profile.copy()
        profile.update( dtype="float32", count=1, compress="lzw", predictor=3, bigtiff="if_safer", tiled=tiled)

        if tiled:
            profile.update(blockxsize=tile_size, blockysize=tile_size)
        else:
            profile.pop("blockxsize", None)
            profile.pop("blockysize", None)

        nodata_in = src.nodata
        nodata_out = np.float32(nodata_in) if nodata_in is not None else None
        profile["nodata"] = nodata_out

        if dst_path.exists():
            raise FileExistsError(dst_path)

        g = np.float32(cal.gain)
        o = np.float32(cal.add_offset)

        with rasterio.open(dst_path, "w", **profile) as dst:
            for _, win in src.block_windows(1):
                dc = src.read(1, window=win).astype(np.float32)
                y = (dc + o) / g
                if nodata_in is not None:
                    y[dc == nodata_in] = nodata_out
                dst.write(y, 1, window=win)

# Transform the per-detector geotif files into a single image for each band          
def merge_geotiffs(src_paths, out_path):
    datasets = [rasterio.open(p) for p in src_paths]
    try:
        mosaic, transform = rio_merge(datasets)
        ref = datasets[0]

        profile = ref.profile.copy()
        profile.update(height=mosaic.shape[1], width=mosaic.shape[2], transform=transform,count=1, 
                       dtype=mosaic.dtype, compress="lzw", predictor=3, bigtiff="if_safer", tiled=False)
        profile.pop("blockxsize", None)
        profile.pop("blockysize", None)

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(mosaic)
    finally:
        for ds in datasets:
            ds.close()
            
# Main runner function
def run_s2_l1b_ds_calibrate_and_merge(
    input_dir, # Containing L1b data
    mtd_xml, # Metadata file for the L1b (needed for gain/offset)
    output_dir, # Where to put merged images
    keep_intermediates=True, # If true, keeps the calibrated per-detector imagery
):
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    by_band = collect_inputs_by_band(input_dir)
    cal_by_band = read_s2_l1b_ds_band_calibration(mtd_xml)

    tmp_obj = None
    if keep_intermediates:
        inter_dir = out_dir / "_calibrated_tiles"
        inter_dir.mkdir(parents=True, exist_ok=True)
    else:
        tmp_obj = tempfile.TemporaryDirectory(prefix="s2l1b_cal_")
        inter_dir = Path(tmp_obj.name)

    outputs = {}
    try:
        for band, det_files in sorted(by_band.items()):
            if band not in cal_by_band:
                raise KeyError(f"No band calibration found in XML for {band}")

            cal = cal_by_band[band]
            calibrated_tiles = []

            for _, src in sorted(det_files):
                dst_tile = inter_dir / f"{src.stem}_CAL.tif"
                if _safe_open_write(dst_tile):
                    if not os.path.exists(dst_tile):
                        calibrate_geotiff(src, dst_tile, cal)
                calibrated_tiles.append(dst_tile)

            last_src = sorted(det_files)[-1][1]
            out_band = out_dir / f"{last_src.stem}_CAL_MERGED.tif"

            if not os.path.exists(out_band):
                merge_geotiffs(calibrated_tiles, out_band)
            outputs[band] = out_band
    finally:
        if tmp_obj is not None:
            tmp_obj.cleanup()

    return outputs


def _process_one_idir(idir: str, keep_intermediates: bool = True):
    """
    Worker function for one DATASTRIP directory.
    Returns a small dict with status + message; exceptions are captured and returned.
    """
    idir = str(idir)
    mtd = glob(f"{idir}/DATASTRIP/*/*S2*.xml")
    if len(mtd) < 1:
        return {"idir": idir, "ok": False, "error": "No metadata"}

    try:
        outputs = run_s2_l1b_ds_calibrate_and_merge(
            input_dir=idir,
            mtd_xml=mtd[0],
            output_dir=f"{idir}/mosaic/",
            keep_intermediates=keep_intermediates,
        )
        return {"idir": idir, "ok": True, "outputs": {k: str(v) for k, v in outputs.items()}}
    except Exception as e:
        return {
            "idir": idir,
            "ok": False,
            "error": str(e),
            "traceback": traceback.format_exc(),
        }


def run_all_idirs_parallel(tdir_inputs, pattern="S2A*", max_workers=1):
    idirs = glob(f"{tdir_inputs}/{pattern}")
    idirs.sort()
    if not idirs:
        raise FileNotFoundError(f"No idirs found under {tdir_inputs} with pattern {pattern!r}")

    if max_workers is None:
        # Conservative default for raster IO: avoid saturating disks.
        # You can override explicitly.
        cpu = os.cpu_count() or 1
        max_workers = min(4, cpu)

    results = []
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        fut_to_idir = {ex.submit(_process_one_idir, idir): idir for idir in idirs}
        for fut in as_completed(fut_to_idir):
            res = fut.result()
            results.append(res)

            # streaming progress
            if res.get("ok"):
                print(f"[ OK] {res['idir']}")
            else:
                print(f"[NOK] {res['idir']}: {res.get('error')}")
    return results

In [3]:
NAME_RX = re.compile(r"_D(?P<det>\d{2})_B(?P<band>\d{2})\.(tif|tiff)$", re.IGNORECASE)

# Mapping between filename band codes and DS metadata bandId integers.
BAND_CODE_TO_ID = {
    "B01": 0,
    "B02": 1,
    "B03": 2,
    "B04": 3,
    "B05": 4,
    "B06": 5,
    "B07": 6,
    "B08": 7,
    "B8A": 8,
    "B09": 9,
    "B10": 10,
    "B11": 11,
    "B12": 12,
}

BAND_ID_TO_CODE = {id: code for code, id in BAND_CODE_TO_ID.items()}

In [ ]:
# Place where the L1b Sentinel-2 data is stored. Each subdir is processed
tdir_inputs = "/data/DATA/"

# Run the processing, can be done in parallel with the `max_workers` parameter.
results = run_all_idirs_parallel(tdir_inputs, pattern="S2A*", max_workers=8)

failed = [r for r in results if not r.get("ok")]
if failed:
    print("\nFailures:")
    for r in failed:
        print(f"- {r['idir']}: {r.get('error')}")

[ OK] /data/DATA/S2A_OPER_PRD_MSIL1B_PDMC_20251205T083018_R092_V20251205T072711_20251205T072722.SAFE
[ OK] /data/DATA/S2A_OPER_PRD_MSIL1B_PDMC_20251205T082608_R070_V20251203T182727_20251203T182742.SAFE
[ OK] /data/DATA/S2A_OPER_PRD_MSIL1B_PDMC_20251205T082704_R072_V20251203T214449_20251203T214500.SAFE
[ OK] /data/DATA/S2A_OPER_PRD_MSIL1B_PDMC_20251208T092516_R095_V20251205T134913_20251205T134924.SAFE
[ OK] /data/DATA/S2A_OPER_PRD_MSIL1B_PDMC_20251205T082509_R073_V20251203T232225_20251203T232240.SAFE
[ OK] /data/DATA/S2A_OPER_PRD_MSIL1B_PDMC_20251205T082914_R085_V20251204T210136_20251204T210158.SAFE
[ OK] /data/DATA/S2A_OPER_PRD_MSIL1B_PDMC_20251208T093543_R099_V20251205T190151_20251205T190202.SAFE
[ OK] /data/DATA/S2A_OPER_PRD_MSIL1B_PDMC_20251205T082801_R085_V20251204T210338_20251204T210400.SAFE
[ OK] /data/DATA/S2A_OPER_PRD_MSIL1B_PDMC_20251208T095109_R099_V20251205T190314_20251205T190340.SAFE
[ OK] /data/DATA/S2A_OPER_PRD_MSIL1B_PDMC_20251208T105434_R110_V20251206T133159_20251206T13